# Scanpy Extra: (For quick testing)

In [37]:
# This cell is labelled 'paramters' to work with papermill remotely
# Papermill will overwite the default local plate value below with whatever is passed to 
# the -p flag in the snakerule shell script
#os.system("conda activate eqtl_study") use this locally if using VScode
plate = 'plate1'
plate = globals().get("plate")
print(f"Processing plate: {plate}")

Processing plate: plate1


### Set Variables

In [ ]:
# Import custom utility packages, lists and functions
import sys
import os
if os.path.exists('/scratch/'):
    root_dir = '/scratch/c.c1477909/eQTL_study_2025/'
else:
    root_dir = '/Users/darren/Desktop/eQTL_study_2025/'
        
sys.path.append(root_dir + 'workflow/scripts/')

from init_env import *
from anndata_utils import *
from gene_lists import *

# Set variables
resolutions = [0.1, 0.2, 0.3] 

### Load L1 data

In [ ]:
# Initialize the environment and get all paths and logger
logger, root_dir, sheets_dir, plate_path, scanpy_dir, report_dir = initialize_env(plate)
logger.info("Loading data ...")
adata = sc.read(scanpy_dir + f'adata_clusters.h5ad')

### Set cluster IDs

In [ ]:
logger.info("Setting cluster IDs ...")
# Set cluster names
cluster_anns = {
    
    '0': 'ExN-UL',
    '1': 'ExN-DL',
    '2': 'RG',
    '3': 'InN',
    '4': 'Endo-Peri',
    '5': 'OPC',
    '6': 'MG'
}

custom_palette = {
    'RG': '#FF5959',
    'ExN-UL': '#00B6EB',
    'InN': '#3CBB75FF',
    'ExN-DL': '#CEE5FD',
    'Endo-Peri': '#B200ED',
    'MG': '#F58231',
    'OPC': '#FDE725FF'
}

final_genes = [
    "CUX2", "SATB2", "PLXNA4", "DSCAML1",         
    "TLE4", "SULF2", "LMO3",                       
    "GAD1", "GAD2", "ERBB4",  "BCL11B",            
    "RELN", "TP73",             
    "GLI3", "GLI2", "PRDM16", "PAX6", "SOX2",      
    "COL4A1", "FN1",                              
    "PDGFRA",                                      
    "C3",                                          
]


# Create a new column in adata.obs with cell type names
adata.obs['cell_type'] = adata.obs['leiden_harmony_0.1'].map(cluster_anns)

### Create UMAP - manuscript

In [ ]:

import matplotlib.gridspec as gridspec

# -----------------------------
# Configuration
# -----------------------------
n_genes = len(final_genes)

# Feature plot grid geometry
ncols = 3
nrows = math.ceil(n_genes / ncols)

# Figure sizing (in inches)
fig_width = 8.0
fig_height = 6.0 + (nrows * 1.2)

# Font control
plt.rcParams.update({
    "font.size": 10,
    "axes.titlesize": 14,
})

# -----------------------------
# Create figure and layout
# -----------------------------
fig = plt.figure(figsize=(fig_width, fig_height))
gs = gridspec.GridSpec(
    nrows=nrows,
    ncols=ncols + 2,      # extra width for Panel A
    figure=fig,
    wspace=0.3,
    hspace=0.4
)

# -----------------------------
# Panel A: Cell type UMAP
# -----------------------------
ax_main = fig.add_subplot(gs[:, :2])

sc.pl.umap(
    adata,
    color="cell_type",
    ax=ax_main,
    legend_loc="on data",
    legend_fontsize=12,
    legend_fontoutline=4,
    size=12,
    frameon=False,
    title="",
    show=False
)

ax_main.set_title("A", loc="left", fontweight="bold", fontsize=16)

# -----------------------------
# Panel B: Feature plots
# -----------------------------
feature_axes = []
for i, gene in enumerate(final_genes):
    row = i // ncols
    col = i % ncols
    ax = fig.add_subplot(gs[row, col + 2])
    feature_axes.append(ax)

    sc.pl.umap(
        adata,
        color=gene,
        ax=ax,
        vmin=0,
        vmax="p99",
        cmap="magma_r",
        size=10,
        frameon=False,
        title=gene,
        show=False
    )

    ax.title.set_fontsize(14)

# Label Panel B
feature_axes[0].set_title("B", loc="left", fontweight="bold", fontsize=16)

# -----------------------------
# Save outputs
# -----------------------------
plot_dir = '../results/13MANUSCRIPT_PLOTS_TABLES/'
plt.savefig(plot_dir + "umaps_manuscript.png", dpi=300, bbox_inches="tight")
plt.savefig(plot_dir + "umaps_manuscript.pdf", bbox_inches="tight")
plt.close(fig)


# Version 2

In [ ]:
# Your genes list (assuming valid_genes is already filtered)

valid_genes = final_genes  # Replace with your actual filtering if needed

logging.info("Creating publication-ready UMAP figure...")

# Create figure with GridSpec: 3 rows, 5 columns (left: wide col for A, right: 4 equal cols for B)
fig = plt.figure(figsize=(14, 6), constrained_layout=True)
gs = GridSpec(3, 5, figure=fig, width_ratios=[3, 1, 1, 1, 1], height_ratios=[1, 1, 1])

# Panel A: Cell-type UMAP (spans full height, left column)
ax_a = fig.add_subplot(gs[0:3, 0])
sc.pl.umap(
    adata,
    color='cell_type',
    add_outline=True,  # Enabled for cluster emphasis; comment out if not wanted
    legend_loc="on data",
    legend_fontsize=12,
    legend_fontoutline=4,
    title='',
    # palette=custom_palette,  # Uncomment and define if needed
    ax=ax_a,
    show=False,
    frameon=False,
    size=5,  # Consistent point size
)

# Panel B: Feature plots in 3x4 grid (right 4 columns, 3 rows)
axes_b = []
for i in range(3):  # Rows
    for j in range(4):  # Columns
        ax = fig.add_subplot(gs[i, j + 1])
        axes_b.append(ax)

# Plot each gene on its ax
for idx, gene in enumerate(valid_genes):
    sc.pl.umap(
        adata,
        color=gene,
        vmin=0,
        vmax="p99",
        cmap="viridis",
        ax=axes_b[idx],
        show=False,
        frameon=False,
        size=5,
        title=gene,  # Gene name as title
        legend_loc=None,  # No legend on data; colorbar handles it
    )
    # Adjust title font for readability
    axes_b[idx].set_title(gene, fontsize=14, fontweight='bold', pad=10)

# Add panel labels
fig.text(0.02, 0.98, 'A', fontsize=18, fontweight='bold', va='top')
fig.text(0.45, 0.98, 'B', fontsize=18, fontweight='bold', va='top')

# Global adjustments for publication quality
for ax in [ax_a] + axes_b:
    ax.tick_params(labelsize=12)  # Larger tick labels
    if hasattr(ax, 'get_legend') and ax.get_legend() is not None:
        for text in ax.get_legend().get_texts():
            text.set_fontsize(12)
    # Adjust colorbar if present (for feature plots)
    if len(ax.get_children()) > 1:  # Check for colorbar
        cbar = ax.get_children()[-1]  # Last child is often the colorbar
        if isinstance(cbar, plt.Colorbar):
            cbar.ax.tick_params(labelsize=10)
            cbar.set_label('Expression', fontsize=12)

plt.tight_layout(pad=0.5, w_pad=0.5, h_pad=0.5)

# Save as 300 DPI PNG and PDF
plt.savefig(plot_dir + 'umap_publication.png', dpi=300, bbox_inches='tight')
plt.savefig(plot_dir + 'umap_publication.pdf', dpi=300, bbox_inches='tight')